In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
import torch.nn.functional as F
import pandas as pd
from torch.amp import autocast, GradScaler

# Set device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Base set up

In [ ]:
MODEL_STRING = "distilbert-base-cased"

In [ ]:
pretrained_model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_STRING,
    num_labels=1,
)

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_STRING)

# Data

In [ ]:
class TMDBReviewDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length):
        self.data = csv_file
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        content = row['content']
        rating = row['rating']

        # Convert rating to 0–1
        target = float(rating) / 10
        # Tokenize
        content_encoding = self.tokenizer(
            content,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Remove the batch dimension
        # Batched by DataLoader...
        item = { key: val.squeeze(0) for key, val in content_encoding.items() }

        # Add labels
        item["labels"] = torch.tensor(target, dtype=torch.float)

        return item

In [ ]:
# load your dataset and set up your dataloader
# If applicable, please include your data with your final submission
# along with instructions for how to load it

data = pd.read_csv("review_data.csv")
dataset = TMDBReviewDataset(data, tokenizer, 256)

# Split Data

total = len(dataset)
train_size = int(.8 * total)
test_size = int(.1 * total)
val_size = int(total - (train_size + test_size))

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(26)
)

BATCH_SIZE = 100

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

### Example Data

In [ ]:
print("TOTAL DATA SIZE:", total)
print("TRAIN SIZE", train_size)
print("TEST SIZE:", test_size)
print("VAL SIZE:", val_size, "\n\n\n")

for i in range(0, 50, 10):
  print("---------CONTENT:", data.iloc[i]["content"])
  print("---------LABEL:", data.iloc[i]["rating"], "\n\n\n")



## Example tokenized data

In [ ]:
data_iter = iter(train_loader)
batch = next(data_iter)
print("RAW BATCH DATA FORMAT:\n", batch)
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]
labels = batch["labels"]
print("\n\n\nROWS:\n\n\n")
for i in range(5):
  print("-------- EMBEDDED CONTENT:", input_ids[i])
  print("-------- ATTENTION MASK:", attention_mask[i])
  print("-------- LABELS:", labels[i], "\n\n\n")

# Implement networks in PyTorch

In [ ]:
def model_accuracy(model, data_loader):
  model.eval()
  total, absolute_errors = 0, 0
  with torch.no_grad():
    for batch in data_loader:
        # batch["input_ids"] = batch["input_ids"].to(device)
        # batch["attention_mask"] = batch["attention_mask"].to(device)
        # batch["labels"] = batch["labels"].to(device)
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model.to(device)(**batch)
        predicted = outputs.logits.squeeze(-1)
        labels = batch["labels"]

        total += labels.size(0)
        absolute_errors += torch.abs(predicted - labels).sum().item()
  return absolute_errors/total

In [ ]:
# to make faster
scaler = GradScaler(device='cuda')

# Load Base model and wrap into a classifier
finetuned_model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_STRING,
    num_labels=1,
).to(device)

optimizer = torch.optim.AdamW(finetuned_model.parameters(), lr = .00001)
criterion = nn.MSELoss()

EPOCHS = 5
epoch_train_mae = []
epoch_val_mae = []

epoch_train_mae.append(model_accuracy(finetuned_model, train_loader))
epoch_val_mae.append(model_accuracy(finetuned_model, val_loader))

for epoch in range(EPOCHS):
  finetuned_model.train()
  epoch_loss = 0

  for batch in train_loader:
    # batch["input_ids"] = batch["input_ids"].to(device)
    # batch["attention_mask"] = batch["attention_mask"].to(device)
    # batch["labels"] = batch["labels"].to(device)
    batch = {k: v.to(device) for k, v in batch.items()}

    optimizer.zero_grad()

    # This speeds up on colab
    with autocast(device_type="cuda"):
      outputs = finetuned_model(**batch)
      pred = outputs.logits.squeeze(-1)
      labels = batch["labels"]
      loss = criterion(pred, labels)
    scaler.scale(loss).backward()
    torch.nn.utils.clip_grad_norm_(finetuned_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    epoch_loss += loss.item()

  avg_loss = epoch_loss / len(train_loader)
  print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss/len(train_loader):.4f}")

  epoch_train_mae.append(model_accuracy(finetuned_model, train_loader))
  epoch_val_mae.append(model_accuracy(finetuned_model, val_loader))

In [ ]:
# Test
print("untrained MAE:",  model_accuracy(pretrained_model, test_loader))
print("finetuned MAE:",  model_accuracy(finetuned_model, test_loader))

In [ ]:
# save model if we want
# finetuned_model.save_pretrained("finetuned_distilbert_regression_model")
# tokenizer.save_pretrained("finetuned_distilbert_regression_model")

In [ ]:
# plot train and validation accuracy of your model during training
epochs = range(len(epoch_train_mae) -1)
plt.plot(epochs, epoch_train_mae[1:], label='Train MAE')
plt.plot(epochs, epoch_val_mae[1:], label='Validation MAE')

plt.title("Train vs Validation MAE per Epoch")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.grid(True)
plt.legend()

plt.show()

### Simple Model Inference

In [ ]:

# A review of The Machinist from someone which gave 3.5 stars -> 7/10
review = "I don’t know if it’s because I’ve seen too many movies with this twist now but I kind of saw it coming from like 10 minutes in. It didn’t really matter though because it was still really well done. I think that the whole theme of guilt eating away at the rest of your life was great. Along with the depiction of psychosis/schizophrenia (I’m not sure which one it would technically be) was top notch, which probably has a lot to do with the actor. Christian Bales performance overall was just amazing, the transformation he went through to is unreal. He is easily one of the best actors of our time, and definitely the most devoted. This movie is worth a watch for Bale alone, and the rest of the movie is just a nice little bonus."

encoding = tokenizer(
    review,
    max_length=256,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

finetuned_model.eval()
with torch.no_grad():
    outputs = finetuned_model(**encoding.to(device))
    prediction = outputs.logits.squeeze().item()

prediction_0_10 = torch.sigmoid(torch.tensor(prediction)).item() * 10
print("Mapped to [0,10]:", prediction_0_10)

# Results

In [ ]:
# diplay test results:
pretrained_model.eval()
untrained_predicted = []
untrained_labels = []
with torch.no_grad():
  for batch in test_loader:
      # batch["input_ids"] = batch["input_ids"].to(device)
      # batch["attention_mask"] = batch["attention_mask"].to(device)
      # batch["labels"] = batch["labels"].to(device)
      batch = {k: v.to(device) for k, v in batch.items()}

      outputs = pretrained_model.to(device)(**batch)
      untrained_predicted += outputs.logits.squeeze(-1).cpu()
      untrained_labels += batch["labels"].cpu()

In [ ]:
# diplay test results:
finetuned_model.eval()
predicted = []
labels = []
with torch.no_grad():
  for batch in test_loader:
      # batch["input_ids"] = batch["input_ids"].to(device)
      # batch["attention_mask"] = batch["attention_mask"].to(device)
      # batch["labels"] = batch["labels"].to(device)
      batch = {k: v.to(device) for k, v in batch.items()}

      outputs = finetuned_model.to(device)(**batch)
      predicted += outputs.logits.squeeze(-1).cpu()
      labels += batch["labels"].cpu()

In [ ]:

plt.scatter(untrained_labels, untrained_predicted)

plt.title("Pretrained Predicted vs True Labels")
y=plt.ylim(0,1)
plt.xlabel("True Labels")
plt.ylabel("Predicted Labels")
plt.grid(True)

plt.show()

plt.scatter(labels, predicted)

plt.title("Finetuned Predicted vs True Labels")
y=plt.ylim(0,1)
plt.xlabel("True Labels")
plt.ylabel("Predicted Labels")
plt.grid(True)

plt.show()

## Measure precision and accuracy


In [ ]:
from scipy.stats import norm

# Treat the prediction errors as a normal distribution:
#   mean error -> accuracy (bias: are we systematically off?)
#   std of error -> precision (how tightly clustered are predictions?)

def error_stats(predicted, labels):
    pred = (torch.stack(predicted) if isinstance(predicted, list) else predicted) * 10
    true = (torch.stack(labels) if isinstance(labels, list) else labels) * 10
    errors = (pred - true).numpy()
    return errors, errors.mean(), errors.std()

untrained_errors, untrained_mean, untrained_std = error_stats(untrained_predicted, untrained_labels)
finetuned_errors, finetuned_mean, finetuned_std = error_stats(predicted, labels)

print(f"Pretrained: mean error = {untrained_mean:+.3f}; std = {untrained_std:.3f}")
print(f"Finetuned: mean error = {finetuned_mean:+.3f}; std = {finetuned_std:.3f}")

for errs, mu, sigma, title in [
    (untrained_errors, untrained_mean, untrained_std, "Pretrained"),
    (finetuned_errors, finetuned_mean, finetuned_std, "Finetuned"),
]:
    plt.hist(errs, bins=40, density=True, alpha=0.6)
    xs = np.linspace(errs.min(), errs.max(), 200)
    plt.plot(xs, norm.pdf(xs, mu, sigma))
    plt.title(f"{title}: mean={mu:+.2f}, std={sigma:.2f}")
    plt.show()


maybe show some example inferences here?